# [Title]

---

## Preparation

- [Github link](google.com) *[Optional]*

- Number of words: ***

- Runtime: *** hours (*Memory 10 GB, CPU Intel i7-10700 CPU @2.90GHz*)

- Coding environment: SDS Docker (or anything else)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/) (or other license that you like).

- Additional library *[libraries not included in SDS Docker or not used in this module]*:
    - **watermark**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.
    - ......

---

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

---

## Introduction

[[ go back to the top ]](#Table-of-contents)

---

## Research questions

[[ go back to the top ]](#Table-of-contents)

---

## Data

[[ go back to the top ]](#Table-of-contents)

### 3.1 Data Sources

| Dataset | Source | Spatial coverage | Temporal coverage |
|---|---|---|---|
| GitHub repository & activity signals | GitHub REST API (metadata, contributors, commit/PR participation where collected) | Global; restricted to prominent candidates and users geocoded into the 148-city list | Event and `created_at` timestamps; monthly aggregation (YYYYMM) in core tables |
| Hugging Face model signals | Hugging Face Hub metadata & author fields | Global; restricted to prominent candidates | Model `created_at`; monthly aggregation where used |
| Curated target cities | In-repository `city_list.csv` (Step 3d) | 148 cities, 50 countries, 9 macro-regions | Static study frame (versioned with pipeline) |
| Prominent project universe | `step1c_merge_candidates.py` → `prominent_projects_master.csv` | Unified GitHub + HF project identifiers | Per-project creation time (ISO 8601) |
| External city covariates | Compiled indicators in `step6_augment_city_attributes.py` (e.g. WB-class proxies, QS counts) | City- or country-level joins to the 148 cities | ~2022–2023 reference values |
| HF derivation edges (optional input to Step 5) | `step5b_build_hf_derivation_edges.py` | Project–project ancestor links | Descendant / ancestor creation months |


### 3.2 Data Preparation

| Category | Source / method | Description & preprocessing |
|---|---|---|
| Prominent project filter | Step 1c merge of `github_candidates` and `hf_candidates` | Retains `prominent_flag = 1` only; unified schema for platforms, metrics, and AI-relevance fields. |
| User–city mapping | `github_owner_locations.csv` + `hf_author_locations.csv` + Steps 3b–3d | Raw location strings cleaned and matched to `city_list`; only target cities enter adoption and edge construction. |
| Adoption events | `step5_build_core_tables.py` | For each (city, project), first linking month (GitHub contributors/participation dates when available; else rule-based fallback); global origin month; non-negative lag; `is_originator` from owner vs contributor logic (plus HF derivation supplement when edges exist). |
| Collaboration edges | Step 5 | Undirected city pairs: shared contributors on the same prominent repo (GitHub) plus distinct HF derivation pairs; `edge_weight` counts shared “project units”; monthly snapshots attribute edges to project months. |
| City attributes | Step 5 + Step 6 | Sums and network centralities from adoption + aggregated edges; Step 6 merges population, GDP, education, internet, R&D, research capacity, timezone, region, and per-capita rates; `cluster` / `role` appended only in later analysis stages. |


### 3.3 Variables Selected

The final analytical dictionaries document **57 variables** in **5 groups** (one table per derived CSV delivered by the pipeline). **Variables selected** for modelling and EDA are drawn from these groups; types are *Binary*, *Integer*, *Continuous*, *Categorical*, and *Ordinal*. File outputs: `data/processed/prominent_projects_master.csv`, `data/output/city_project_adoption_events.csv`, `data/output/city_attributes.csv`, `data/output/city_collaboration_edges.csv`, `data/output/city_collaboration_edges_monthly.csv`.


**Prominent projects master (15)**

| Variable | Type | Definition |
|---|---|---|
| `project_id` | Categorical | Unified project identifier (e.g. gh_*, hf_*). |
| `platform` | Categorical | GitHub or HuggingFace. |
| `full_id` | Categorical | Native platform string (full repo name or HF model id). |
| `project_name` | Categorical | Short display name. |
| `hf_type` | Categorical | Hugging Face resource type (empty for GitHub). |
| `tags` | Categorical | Tag string (often semicolon-separated). |
| `metric_stars` | Integer | Star count (GitHub; often missing for HF). |
| `metric_forks` | Integer | Fork count (GitHub; often missing for HF). |
| `metric_downloads` | Integer | Download count (HF; often missing for GitHub). |
| `metric_likes` | Integer | Like count (HF; often missing for GitHub). |
| `created_at` | Categorical | Repository/model creation time (ISO 8601 string). |
| `open_ai_related` | Binary | Flag: project passes open-AI relevance screen. |
| `ai_evidence` | Categorical | Keywords or labels supporting AI relevance. |
| `ai_confidence` | Ordinal | Confidence tier of relevance rule (e.g. high, medium, low). |
| `prominent_flag` | Binary | Retained rows have prominent_flag = 1 only. |

**City–project adoption events (6)**

| Variable | Type | Definition |
|---|---|---|
| `city` | Categorical | City name. |
| `project_id` | Categorical | Project key (GitHub owner/repo or Hugging Face hf_* id). |
| `global_origin_month` | Integer | Global project origin month as YYYYMM. |
| `city_first_adoption_month` | Integer | First month the city links to the project (YYYYMM). |
| `lag` | Integer | Months from global_origin_month to city_first_adoption_month (non-negative). |
| `is_originator` | Binary | 1 if the city originated the project; 0 otherwise. |

**City-level attributes (28)**

| Variable | Type | Definition |
|---|---|---|
| `city` | Categorical | Standardised city name; joins to city_list.matched_city. |
| `country` | Categorical | Country name. |
| `lat` | Continuous | Latitude of city centroid (degrees). |
| `lon` | Continuous | Longitude of city centroid (degrees). |
| `entity_count` | Integer | Count of GitHub/HF entities linked to the city in the study frame (activity denominator from city_list). |
| `origination_count` | Integer | Number of adoption events where the city is project originator (is_originator = 1). |
| `origination_rate` | Continuous | origination_count divided by entity_count. |
| `adoption_count` | Integer | Distinct project_id with at least one adoption event. |
| `adoption_rate` | Continuous | adoption_count divided by entity_count. |
| `avg_lag` | Continuous | Mean adoption lag in months across all city–project events. |
| `median_lag` | Continuous | Median adoption lag in months. |
| `collaboration_count` | Integer | Sum of edge_weight on all incident collaboration edges (undirected). |
| `weighted_degree` | Continuous | Weighted degree on the collaboration graph (equivalent to collaboration_count here). |
| `betweenness` | Continuous | Weighted betweenness centrality. |
| `eigenvector_centrality` | Continuous | Weighted eigenvector centrality (zero-filled if estimation fails). |
| `population_million` | Continuous | Metro population in millions (Step 6 external lookup). |
| `gdp_per_capita` | Continuous | Country GDP per capita in current USD (~2023, Step 6). |
| `education_tertiary_pct` | Continuous | Country tertiary gross enrolment ratio (%). |
| `internet_users_pct` | Continuous | Country internet users as share of population (%). |
| `rd_expenditure_pct` | Continuous | Country R&D expenditure as % of GDP. |
| `research_capacity` | Integer | Count proxy for local research capacity (e.g. QS top-500 universities in city). |
| `timezone_utc` | Continuous | Approximate UTC offset in hours (0.5 h grid from longitude). |
| `region` | Categorical | Macro-region (e.g. East Asia, Europe, North America). |
| `origination_rate_pop` | Continuous | Origination events per million population. |
| `adoption_rate_pop` | Continuous | Adopted distinct projects per million population. |
| `collaboration_rate_pop` | Continuous | Collaboration weight sum per million population. |
| `cluster` | Ordinal | Cluster label from K-means (or similar); appended in later analysis stages. |
| `role` | Categorical | Interpretable role name for cluster (e.g. Global Innovation Hub). |

**City-pair collaboration edges, aggregated (4)**

| Variable | Type | Definition |
|---|---|---|
| `source_city` | Categorical | Lexicographically smaller endpoint of the undirected pair. |
| `target_city` | Categorical | Other endpoint of the undirected pair. |
| `edge_weight` | Integer | Count of shared project units linking the pair (GitHub co-repo; HF one unit per derivation edge). |
| `shared_projects` | Integer | In this build, incremented with edge_weight (numerically identical). |

**City-pair collaboration edges, monthly snapshots (4)**

| Variable | Type | Definition |
|---|---|---|
| `source_city` | Categorical | Same semantics as aggregated edge table. |
| `target_city` | Categorical | Same semantics as aggregated edge table. |
| `month` | Integer | Attribution month YYYYMM. |
| `edge_weight` | Binary | 1 if at least one collaboration is attributed in that month for the city pair. |

### 3.4 Data Quality

| Stage | Records | Notes |
|---|---|---|
| Curated city list (`city_list.csv`) | 148 cities | High-confidence geocoding / matching (50 countries, 9 macro-regions). |
| `prominent_projects_master.csv` | 23,481 projects | Prominent GitHub + HF rows (excluding header). |
| `city_project_adoption_events.csv` | 39,158 events | City–project adoption / origin rows. |
| `city_attributes.csv` | 148 rows × 28 columns | One row per city; Step 6 `population_million` missing for 5 cities (~3.4%) in published diagnostics. |
| `city_collaboration_edges.csv` | 9,636 edges | Aggregated undirected city pairs. |
| `city_collaboration_edges_monthly.csv` | 98,147 rows | Month × pair snapshots (excluding header). |
| Join keys | — | `project_id` links events to the master table; `city` links events and attributes; edge tables use paired `city` names consistent with `city_list`. |


---

## Methodology

[[ go back to the top ]](#Table-of-contents)

*[Note: a flow chart that describes the methodology is strongly encouraged - see the example below. This flow chart can be made using Microsoft powerpoint or visio or other software]*

Source: see [link](https://linkinghub.elsevier.com/retrieve/pii/S2210670722004437).

![image.png](attachment:image.png)

---

## Results and discussion

[[ go back to the top ]](#Table-of-contents)

---

## Conclusion

[[ go back to the top ]](#Table-of-contents)

---

## References

[[ go back to the top ]](#Table-of-contents)